# Import bibliotek

In [32]:
import requests
import json
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field

load_dotenv()

True

In [23]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

In [27]:
system_message = """
Dokonujesz oceny rysyka da wędróki górskiej dla wskazanego punktu w Tatrach.
Ocenę wykonujesz na podstawie 24-godzinnej prognozy (prognozowana temperatura co 3 godziny).
W odpowiedzi wskazujesz swoją ocenę poziomu ryzyka oraz krótkie uzasadnienie.
"""

In [28]:
with open("../data/json/20260408_220641.json", encoding="utf-8") as f:
    data = json.load(f)
forecast_dict = dict(list(data[0]["temperatures"].items())[:8])

In [36]:
messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": json.dumps(forecast_dict)}
]

In [33]:
class RiskAssessment(BaseModel):
    recommendation: str = Field(description="Wartość z listy: safe, risky, dangerous")
    justification: list[str] = Field(description="Krótkie uzasadnienie rekomendacji")

In [37]:
response = client.responses.parse(
    model="gpt-4o-mini",
    input=messages,
    text_format=RiskAssessment
)

In [38]:
response.output_parsed.model_dump()

{'recommendation': 'dangerous',
 'justification': ['Prognozowane temperatury w zakresie -8.43 do -2.79°C mogą powodować hipotermię.',
  'W nocy i nad ranem spodziewane są najniższe temperatury, co zwiększa ryzyko niskotemperaturowych kontuzji i zagrożeń.',
  'Stale ujemne temperatury przez całą dobę są niebezpieczne dla turystów, którzy nie są odpowiednio przygotowani.']}